> This notebooks is entirely about entity matching part of  Web-Data Integration Pipeline. It's more of a exploration rather than integration for entity-matching.

> We'll experiment and ultimately decide on which methods to implement in the final pipeline in this notebook.

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
import logging

LOGS = OUTPUT_DIR / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler(LOGS / 'pydi.log'),
          logging.StreamHandler()
      ],
    force=True
)

## Basic Dataset Summary

In [4]:
datasets = [amazon_books, goodreads, metabooks]
names = ["Amazon", "Goodreads", "Metabooks"]

for df, name in zip(datasets, names):
    print(f"{name}:")
    print(f"  Records: {len(df):,}")
    print(f"  Attributes: {len(df.columns)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dataset name: {df.attrs.get('dataset_name', 'unknown')}")
    print()

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Amazon:
  Records: 271,360
  Attributes: 8
  Columns: ['id', 'title', 'author', 'publish_year', 'publisher', 'isbn_clean', 'title_norm', 'author_norm']
  Dataset name: amazon_books

Goodreads:
  Records: 52,478
  Attributes: 16
  Columns: ['id', 'title', 'author', 'rating', 'rating_number', 'language', 'genres', 'book_format', 'edition', 'page_count', 'publisher', 'publish_year', 'price', 'isbn_clean', 'title_norm', 'author_norm']
  Dataset name: goodreads

Metabooks:
  Records: 535,159
  Attributes: 14
  Columns: ['id', 'title', 'author', 'rating', 'rating_number', 'language', 'genres', 'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean', 'title_norm', 'author_norm']
  Dataset name: metabooks

Total records across all datasets: 858,997


In [5]:
random_common_book=list(set(amazon_books.isbn_clean)&set(goodreads.isbn_clean)&set(metabooks.isbn_clean))[60]
display(amazon_books[amazon_books.isbn_clean==random_common_book])
display(goodreads[goodreads.isbn_clean==random_common_book])
display(metabooks[metabooks.isbn_clean==random_common_book])

,id,title,author,publish_year,publisher,isbn_clean,title_norm,author_norm
108549,amazon_108550,A Man Called Peter: The Story of Peter Marshall,Catherine Marshall,2002,Chosen Books,0800793110,a man called peter the story of peter marshall,catherine marshall


,id,title,author,rating,rating_number,language,genres,book_format,edition,page_count,publisher,publish_year,price,isbn_clean,title_norm,author_norm
17477,goodreads_17478,A Man Called Peter: The Story of Peter Marshall,Catherine Marshall,4.27,9748,English,"['Biography', 'Nonfiction', 'Christian', 'Reli...",Paperback,nan,384,Chosen Books,2002,5.61,0800793110,a man called peter the story of peter marshall,catherine marshall


,id,title,author,rating,rating_number,language,genres,publisher,page_count,price,publish_year,isbn_clean,title_norm,author_norm
466385,metabooks_466386,A Man Called Peter: The Story of Peter Marshall,Catherine Marshall,4.6,169,English,"['Biographies', 'Memoirs', 'Community', 'Cultu...",Chosen Books,366,14.98,<NA>,0800793110,a man called peter the story of peter marshall,catherine marshall


# Entity Matching

## Blocking

> In this part we are diving deep into blocking methods to see which one performs the best. Then we are gonna pick that one for rule-based matching.

In [6]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import SortedNeighbourhoodBlocker
from PyDI.entitymatching import TokenBlocker
from PyDI.entitymatching import EmbeddingBlocker
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 200)

from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.io import load_csv

### Metabooks - Amazon

In [7]:
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

BLOCK_EVAL_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
st_blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a = st_blocker_m2a.materialize()


st_blocker_m2a_norm = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a_norm = st_blocker_m2a_norm.materialize()

sn_blocker_m2a_20 = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title_norm',
    window=20,
    batch_size=750,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2a = sn_blocker_m2a_20.materialize()

sn_blocker_m2a_25 = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title_norm',
    window=25,
    batch_size=750,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2a_25 = sn_blocker_m2a_25.materialize()

token_blocker_m2a = TokenBlocker(
    metabooks, amazon_books,
    column='title_norm',
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=3,
    ngram_type='character'
)
#token_candidates_m2a = token_blocker_m2a.materialize()



embedding_blocker_m2a = EmbeddingBlocker(
    metabooks, amazon_books,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=10,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
#embedding_candidates_m2a = embedding_blocker_m2a.materialize()

#standard_candidates_m2a.to_csv(CORR_DIR / "StandardBlocker-Corr-MA.csv", index=False)
#sn_candidates_m2a.to_csv(CORR_DIR / "SNBlocker-Corr-MA.csv", index=False)
#token_candidates_m2a.to_csv(CORR_DIR / "TokenBlocker-Corr-MG.csv", index=False)
#embedding_candidates_m2a.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MA.csv", index=False)




[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 517669 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 245803 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 22962 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 507818 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 236280 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 31220 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurc

In [ ]:
evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
m2a_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2a, st_blocker_m2a],
    'StandardBlockingNorm': [standard_candidates_m2a_norm, st_blocker_m2a_norm],
    'SortedNeighbourhoodBlocker20': [sn_candidates_m2a, sn_blocker_m2a_20],
    'SortedNeighbourhoodBlocker25': [sn_candidates_m2a, sn_blocker_m2a_25],
    #'TokenBlocking': [token_candidates_m2a,token_blocker_m2a],
    #'EmbeddingBlocking': [embedding_candidates_m2a, embedding_blocker_m2a]
}

m2a_correspondences = load_csv(
    MLDS_DIR/"train_MA.csv",
    name="m2a_train", header=1, names=['id1', 'id2', 'label'], add_index=False
)

m2a_results = []
for method_name, candidates in m2a_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2a_correspondences,candidates[1], out_dir=BLOCK_EVAL_DIR)
    result['method'] = method_name
    result['dataset'] = 'm2a'
    m2a_results.append(result)

m2a_best = max(m2a_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for a2g: {m2a_best['method']} (PC: {m2a_best['pair_completeness']:.3f}, RR: {m2a_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.470
[INFO ] root -   Pair Quality:      0.002
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 77/164
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.567
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 93/164
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.927
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 152/164
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.927
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 152/164
[INFO ] root - Blocking evaluation complete!


Best blocking for a2g: SortedNeighbourhoodBlocker20 (PC: 0.927, RR: 1.000)


In [15]:
import pandas as pd

m2a_summary = pd.DataFrame(
    [
        {
            "method": res["method"],
            "pair_completeness": res["pair_completeness"],
            "reduction_ratio": res["reduction_ratio"],
        }
        for res in m2a_results 
    ]
)
m2a_summary

,method,pair_completeness,reduction_ratio
0,StandardBlocking,0.469512,1.000000
1,StandardBlockingNorm,0.567073,0.999999
2,SortedNeighbourhoodBlocker20,0.926829,0.999954
3,SortedNeighbourhoodBlocker25,0.926829,0.999954


### Metabooks - Goodreads Pair

In [17]:
st_blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['title','author'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
standard_candidates_m2g = st_blocker_m2g.materialize()

st_blocker_m2g_norm = StandardBlocker(
    metabooks, goodreads,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
standard_candidates_m2g_norm = st_blocker_m2g_norm.materialize()

sn_blocker_m2g_20 = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2g = sn_blocker_m2g_20.materialize()

sn_blocker_m2g_25 = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title_norm',
    window=25,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2g_25 = sn_blocker_m2g_25.materialize()

token_blocker_m2g = TokenBlocker(
    metabooks, goodreads,
    column='title_norm',
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=2,
    ngram_type='character'
)
#token_candidates_m2g = token_blocker_m2g.materialize()


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks, goodreads,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
#embedding_candidates_m2g = embedding_blocker_m2g.materialize()

#standard_candidates_m2g.to_csv(CORR_DIR / "StandardBlocker-Corr-MG.csv", index=False)
#sn_candidates_m2g.to_csv(CORR_DIR / "SNBlocker-Corr-MG.csv", index=False)
#token_candidates_m2g.to_csv(CORR_DIR / "TokenBlocker-Corr-MG.csv", index=False)
#embedding_candidates_m2g.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MG.csv", index=False)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 518708 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 52384 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4112 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 517669 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 51247 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4262 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanme

In [ ]:
evaluator = EntityMatchingEvaluator()

m2g_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2g, st_blocker_m2g],
    'StandardBlockingNorm': [standard_candidates_m2g_norm, st_blocker_m2g_norm],
    'SortedNeighbourhoodBlocker20': [sn_candidates_m2g, sn_blocker_m2g_20],
    'SortedNeighbourhoodBlocker25': [sn_candidates_m2g_25, sn_blocker_m2g_25],
    #'TokenBlocking': [token_candidates_m2g,token_blocker_m2g],
    #'EmbeddingBlocking': [embedding_candidates_m2g, embedding_blocker_m2g]
}

m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_train",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

m2g_results = []
for method_name, (cand_df, blocker) in m2g_blocking_candidates.items():
    result = evaluator.evaluate_blocking(
        cand_df,
        m2g_correspondences,
        blocker,
        out_dir=BLOCK_EVAL_DIR
    )
    result['method'] = method_name
    result['dataset'] = 'm2g'
    m2g_results.append(result)

m2g_best = max(m2g_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for m2g: {m2g_best['method']} (PC: {m2g_best['pair_completeness']:.3f}, RR: {m2g_best['reduction_ratio']:.3f})")


[INFO ] root -   Pair Completeness: 0.188
[INFO ] root -   Pair Quality:      0.006
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 33/176
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.188
[INFO ] root -   Pair Quality:      0.002
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 33/176
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.932
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 164/176
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.932
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 164/176
[INFO ] root - Blocking evaluation complete!


Best blocking for m2g: SortedNeighbourhoodBlocker20 (PC: 0.932, RR: 1.000)


In [26]:
m2g_summary = pd.DataFrame(
    [
        {
            "method": res["method"],
            "pair_completeness": res["pair_completeness"],
            "reduction_ratio": res["reduction_ratio"],
        }
        for res in m2g_results 
    ]
)
m2g_summary

,method,pair_completeness,reduction_ratio
0,StandardBlocking,0.187500,1.000000
1,StandardBlockingNorm,0.187500,0.999999
2,SortedNeighbourhoodBlocker20,0.931818,0.999936
3,SortedNeighbourhoodBlocker25,0.931818,0.999920


## Rule-Based Entity Matching

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher, EntityMatchingEvaluator
from PyDI.io import load_csv
from pathlib import Path

matcher = RuleBasedMatcher()
OUTPUT_DIR = Path("output/entity-matching")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_matching(combo_key, left_df, right_df, candidates, comparators, weights,
                 threshold, test_file, id_column="id"):
    correspondences = matcher.match(
        df_left=left_df,
        df_right=right_df,
        candidates=candidates,
        comparators=comparators,
        weights=weights,
        threshold=threshold,
        id_column=id_column,
    )

    test_pairs = load_csv(
        MLDS_DIR / test_file,
        name=f"test_{combo_key.lower()}",
        header=1,
        names=["id1", "id2", "label"],
        add_index=False,
    )

    out_dir = OUTPUT_DIR / combo_key.lower()
    out_dir.mkdir(parents=True, exist_ok=True)

    results = EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences,
        test_pairs=test_pairs,
        out_dir=out_dir,
    )
    results["combo"] = combo_key
    return results

# Base comparators used by both combos
comparators_base = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_ma = comparators_base
comparators_mg = comparators_base + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
]

results_ma = run_matching(
    combo_key="MA",
    left_df=metabooks,
    right_df=amazon_books,
    candidates=sn_blocker_m2a_20,
    comparators=comparators_ma,
    weights=[0.7, 0.2, 0.1],
    threshold=0.7,
    test_file="train_MA.csv",
)

results_mg = run_matching(
    combo_key="MG",
    left_df=metabooks,
    right_df=goodreads,
    candidates=sn_blocker_m2g_20,
    comparators=comparators_mg,
    weights=[0.6, 0.2, 0.05, 0.05, 0.05, 0.05],
    threshold=0.7,
    test_file="train_MG.csv",


[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 535159 x 271360 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 535159 x 271360 elements after 0:00:3.744; 6731058 blocked pairs (reduction ratio: 0.9999536494738233)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:1030.988; found 109696 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  127
[INFO ] root -   True Negatives:  628
[INFO ] root -   False Positives: 7
[INFO ] root -   False Negatives: 37
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.945
[INFO ] root -   Precision: 0.948
[INFO ] root -   Recall:    0.774
[INFO ] root -   F1-Score:  0.852
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 535159 x 52478 el

TypeError: cannot concatenate object of type '<class 'dict'>'; only Series and DataFrame objs are valid

## ML- Based Entity Matching

In [35]:
from PyDI.io import load_parquet
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator


m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)
m2a_correspondences = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

comparators_m2a = [
    # Title similarity
    StringComparator(column='title_norm',similarity_function='jaccard'),
    StringComparator(column='title_norm',similarity_function='cosine'),
    StringComparator(column='title_norm',similarity_function='jaro'),
    StringComparator(column='title_norm',similarity_function='jaro_winkler'),
    
    # author similarity
    StringComparator(column='author_norm',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author_norm',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author_norm',similarity_function='jaro', preprocess=str.lower),
    # publish year similarity
    NumericComparator(column='publish_year',max_difference=1)
    ]
comparators_m2g = comparators_m2a + [
    StringComparator(column='genres',similarity_function='jaccard',preprocess=str.lower,list_strategy='concatenate'),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
]

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)


train_features_m2a = feature_extractor_m2a.create_features(
    metabooks, amazon_books, m2a_correspondences[['id1', 'id2']], labels=m2a_correspondences['label'], id_column='id'
)

train_features_m2g = feature_extractor_m2g.create_features(
    metabooks, goodreads, m2g_correspondences[['id1', 'id2']], labels=m2g_correspondences['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

[INFO ] root - Label distribution: 164 positive, 635 negative
[INFO ] root - Label distribution: 176 positive, 622 negative


In [37]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [39]:
best_models

[LogisticRegression(C=1, max_iter=1000, random_state=42, solver='liblinear'),
 LogisticRegression(C=10, max_iter=1000, random_state=42, solver='liblinear')]

In [40]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import MLBasedMatcher


ml_matcher_m2a = MLBasedMatcher(feature_extractor_m2a)
ml_matcher_m2g = MLBasedMatcher(feature_extractor_m2g)

correspondences_m2a_ml = ml_matcher_m2a.match(
    metabooks, amazon_books,
    candidates=sn_blocker_m2a_20,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g_ml = ml_matcher_m2g.match(
    metabooks, goodreads,
    candidates=sn_blocker_m2g_20,
    id_column='id',
    trained_classifier=best_models[1]
)

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a_ml,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g_ml,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 535159 x 271360 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 535159 x 271360 elements after 0:00:2.176; 6731058 blocked pairs (reduction ratio: 0.9999536494738233)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:2029.259; found 147758 correspondences.
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 535159 x 52478 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 535159 x 52478 elements after 0:00:2.339; 1797032 blocked pairs (reduction ratio: 0.9999360124175761)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:647.147; found 35593 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  125
[INFO ]

{'precision': 0.9084507042253521,
 'recall': 0.7329545454545454,
 'f1': 0.8113207547169812,
 'accuracy': 0.924812030075188,
 'true_positives': 129,
 'false_positives': 13,
 'false_negatives': 47,
 'true_negatives': 609,
 'threshold_used': 0.0,
 'total_correspondences': 35593,
 'filtered_correspondences': 35593,
 'evaluation_timestamp': '2025-11-04T13:44:32.293407',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}